# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [ ]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [3]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # apply the low-rank update (batch, in_dim) -> (batch, out_dim)
        x = self.alpha * (x @ self.A) @ self.B
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 128
out_dim = 64
rank = 8
alpha = 8

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
# e.g., torch.randn(batch, in_dim)
x = torch.randn(16, in_dim)

print(x)
print(layer)
print("Original output:", layer(x))

tensor([[ 0.5229,  0.1553,  0.5247,  ...,  0.9427,  1.1394,  1.0076],
        [-1.0693,  0.4660,  0.7012,  ..., -0.2004,  0.8093, -0.6667],
        [-0.3162,  1.2700, -0.0903,  ...,  1.8661, -2.1111, -0.1183],
        ...,
        [ 1.4663,  0.3888,  0.5244,  ..., -1.8122, -2.5448, -0.5135],
        [ 1.8056, -1.0064,  0.1582,  ..., -0.4569,  0.3888,  1.3338],
        [-0.1424,  0.0666,  0.9221,  ...,  0.2379, -1.1839, -0.3179]])
LoRALayer()
Original output: tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], grad_fn=<MmBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [6]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
# wrap `base_linear` with rank/alpha values
layer_lora_1 = LinearWithLoRA(base_linear, rank=4, alpha=8)
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[-0.0484,  0.6586, -0.2561,  ..., -0.3786, -0.8724, -0.3635],
        [ 1.2142,  0.2975,  0.3639,  ...,  0.7663, -0.4720, -0.5339],
        [ 0.1154,  0.1466, -1.0498,  ...,  0.7262, -1.1158,  0.2073],
        ...,
        [-0.0224, -0.4564,  0.2380,  ..., -0.8651, -0.6700, -0.2480],
        [-0.0197, -1.0699, -0.1417,  ..., -0.6832,  0.1298, -1.3834],
        [ 0.6063,  0.1012,  0.6096,  ...,  1.7777,  0.4449, -0.0407]],
       grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [10]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=4, num_classes=2)
sample_input = torch.randn(16, 4)

with torch.no_grad():
    baseline_output = single_net(sample_input)

single_net.layer = LinearWithLoRA(single_net.layer,rank,alpha)  # replace with LinearWithLoRA

with torch.no_grad():
    lora_output = single_net(sample_input)

print("Outputs match before training?", torch.equal(baseline_output, lora_output))

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [14]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )
        self.alpha = alpha

    def forward(self, x):
        lora_delta = self.lora.A @ self.lora.B  # Shape: (in_dim, out_dim)
        # merge original weights + scaled LoRA correction
        combined_weight = self.linear.weight + self.alpha * lora_delta.T
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(base_linear, rank=4, alpha=8)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[-0.0484,  0.6586, -0.2561,  ..., -0.3786, -0.8724, -0.3635],
        [ 1.2142,  0.2975,  0.3639,  ...,  0.7663, -0.4720, -0.5339],
        [ 0.1154,  0.1466, -1.0498,  ...,  0.7262, -1.1158,  0.2073],
        ...,
        [-0.0224, -0.4564,  0.2380,  ..., -0.8651, -0.6700, -0.2480],
        [-0.0197, -1.0699, -0.1417,  ..., -0.6832,  0.1298, -1.3834],
        [ 0.6063,  0.1012,  0.6096,  ...,  1.7777,  0.4449, -0.0407]],
       grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [23]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        # Flatten the input tensor from (batch, 1, 28, 28) to (batch, 784)
        x = x.view(x.size(0), -1)
        x = self.layers(x)
        return x

In [24]:
# Architecture
num_features = 28 * 28  # MNIST images are 28x28 pixels
num_hidden_1 = 8
num_hidden_2 = 8
num_classes = 10        # MNIST has 10 classes (digits 0-9)

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.01
num_epochs = 15

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [17]:
BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 136MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 63.3MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 73.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.1MB/s]


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [18]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred +=  (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

## Training

In [19]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [25]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

/tmp/ipykernel_1669/2143974910.py:18: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/015|Batch 000/938| Loss: 2.3042
Epoch: 001/015|Batch 400/938| Loss: 0.2319
Epoch: 001/015|Batch 800/938| Loss: 0.6123
Epoch: 001/015 training accuracy: 89.46%
Time elapsed: 0.26 min
Epoch: 002/015|Batch 000/938| Loss: 0.1548
Epoch: 002/015|Batch 400/938| Loss: 0.2698
Epoch: 002/015|Batch 800/938| Loss: 0.2511
Epoch: 002/015 training accuracy: 90.93%
Time elapsed: 0.51 min
Epoch: 003/015|Batch 000/938| Loss: 0.2608
Epoch: 003/015|Batch 400/938| Loss: 0.2750
Epoch: 003/015|Batch 800/938| Loss: 0.1488
Epoch: 003/015 training accuracy: 92.16%
Time elapsed: 0.76 min
Epoch: 004/015|Batch 000/938| Loss: 0.1846
Epoch: 004/015|Batch 400/938| Loss: 0.2809
Epoch: 004/015|Batch 800/938| Loss: 0.2389
Epoch: 004/015 training accuracy: 92.34%
Time elapsed: 1.02 min
Epoch: 005/015|Batch 000/938| Loss: 0.2567
Epoch: 005/015|Batch 400/938| Loss: 0.2409
Epoch: 005/015|Batch 800/938| Loss: 0.1662
Epoch: 005/015 training accuracy: 92.16%
Time elapsed: 1.29 min
Epoch: 006/015|Batch 000/938| Loss:

# Replacing Linear with LoRA Layers

In [26]:
model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=8, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=8, out_features=8, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=8, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model:91.38%
Test accuracy LoRA model:91.38%


## Freezing the Original Linear Layers

In [27]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.0.linear.weight:False
layers.0.linear.bias:False
layers.0.lora.A:True
layers.0.lora.B:True
layers.2.linear.weight:False
layers.2.linear.bias:False
layers.2.lora.A:True
layers.2.lora.B:True
layers.4.linear.weight:False
layers.4.linear.bias:False
layers.4.lora.A:True
layers.4.lora.B:True


In [28]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Epoch: 001/015|Batch 000/938| Loss: 0.3244
Epoch: 001/015|Batch 400/938| Loss: 0.2352
Epoch: 001/015|Batch 800/938| Loss: 0.1735
Epoch: 001/015 training accuracy: 93.01%
Time elapsed: 0.27 min
Epoch: 002/015|Batch 000/938| Loss: 0.2187
Epoch: 002/015|Batch 400/938| Loss: 0.3249
Epoch: 002/015|Batch 800/938| Loss: 0.2295
Epoch: 002/015 training accuracy: 93.12%
Time elapsed: 0.53 min
Epoch: 003/015|Batch 000/938| Loss: 0.2034
Epoch: 003/015|Batch 400/938| Loss: 0.2221
Epoch: 003/015|Batch 800/938| Loss: 0.2732
Epoch: 003/015 training accuracy: 93.24%
Time elapsed: 0.79 min
Epoch: 004/015|Batch 000/938| Loss: 0.1700
Epoch: 004/015|Batch 400/938| Loss: 0.1211
Epoch: 004/015|Batch 800/938| Loss: 0.1174
Epoch: 004/015 training accuracy: 93.04%
Time elapsed: 1.06 min
Epoch: 005/015|Batch 000/938| Loss: 0.3919
Epoch: 005/015|Batch 400/938| Loss: 0.1370
Epoch: 005/015|Batch 800/938| Loss: 0.0641
Epoch: 005/015 training accuracy: 93.15%
Time elapsed: 1.33 min
Epoch: 006/015|Batch 000/938| Loss: